# Task 4 — Ranking: Feature Engineering + LightGBM LambdaRank

This notebook implements the ranking stage of the Steam recommendation pipeline.

**Goal:** Train a LightGBM LambdaRank reranker on retrieval candidates to improve
recommendation quality beyond raw retrieval scores.

**Pipeline:**
1. Load processed data splits (train / val / test / catalog)
2. Simulate retrieval candidates (placeholder until two-tower retriever is ready)
3. Engineer 14 features (user, item, cross, temporal)
4. Train LightGBM LambdaRank with early stopping
5. Evaluate NDCG@10 + Catalog Coverage
6. Analyze feature importance
7. Export model for API serving

**14 Features from Steam columns:**

| Group | Features |
|-------|----------|
| User (4) | activity_count, avg_hours, distinct_items, positive_rate |
| Item (5) | popularity, avg_hours, positive_rate, avg_text_length, is_early_access |
| Cross (3) | category_affinity, hours_vs_avg, popularity_rank |
| Temporal (2) | days_since_last, hour_sin/cos |

**Design decision — Candidate simulation:** Because the retrieval stage (MF-BPR / two-tower)
is being developed in parallel, we simulate retrieval candidates by sampling 1 positive + 99
negative items per user. This mimics what a real FAISS top-100 retrieval would produce.
Once the retriever is ready, we replace this section with actual FAISS candidates.

## 0. Setup & Imports

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from steam_recsys.ranking.features import build_features
from steam_recsys.ranking.train import (
    prepare_ranking_data,
    train_ranker,
    evaluate_ranker,
    save_model,
)

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("data/processed/steam")
MODEL_DIR = Path("models")
ARTIFACT_DIR = Path("artifacts")

print("Imports OK")

## 1. Load Processed Data

These files were produced by Task 1 (Kim Tan). The split is time-based (chronological):
train (earliest 72%) → val (next 8%) → test (most recent 20%).

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "validation.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")
catalog = pd.read_parquet(DATA_DIR / "items.parquet")

print(f"Train:  {len(train):>10,} interactions | {train['user_id'].nunique():>8,} users | {train['item_id'].nunique():>8,} items")
print(f"Val:    {len(val):>10,} interactions | {val['user_id'].nunique():>8,} users | {val['item_id'].nunique():>8,} items")
print(f"Test:   {len(test):>10,} interactions | {test['user_id'].nunique():>8,} users | {test['item_id'].nunique():>8,} items")
print(f"Catalog: {len(catalog):>8,} items")
print(f"\nSparsity: {1 - len(train) / (train['user_id'].nunique() * train['item_id'].nunique()):.4%}")
print(f"Positive rate: {train['is_positive'].mean():.2%} (hours >= 1.0)")

## 2. Simulate Retrieval Candidates

**Why simulate?** The MF-BPR and two-tower retrievers are being built in Task 3.
To avoid blocking, we generate placeholder candidates:

- For each user in val/test, pick 1 known positive item (ground truth)
- Add 99 random negative items (not interacted with)
- This mirrors the shape of a real FAISS top-100 retrieval

**Replace this section** when the retriever exports real FAISS top-100 per user.

In [ ]:
def simulate_candidates(
    interactions: pd.DataFrame,
    train: pd.DataFrame,
    catalog: pd.DataFrame,
    n_negatives: int = 99,
    seed: int = 42,
) -> pd.DataFrame:
    """Simulate retrieval candidates: 1 positive + N negatives per user.

    This mimics a retriever's top-100 output. Replace with real FAISS results
    once the two-tower model is ready.
    """
    rng = np.random.default_rng(seed)
    all_items = set(catalog["item_id"].unique())

    rows = []
    for user_id, group in interactions.groupby("user_id"):
        # User's training items (to exclude from negatives)
        train_user_items = set(
            train[train["user_id"] == user_id]["item_id"].unique()
        )
        # User's test-period positive items
        positive_items = set(group[group["is_positive"]]["item_id"].unique())

        if not positive_items:
            continue

        # Pick 1 positive item
        pos_item = rng.choice(list(positive_items))
        pos_row = group[group["item_id"] == pos_item].iloc[0]
        rows.append({
            "user_id": user_id,
            "item_id": pos_item,
            "hours": pos_row["hours"],
            "event_time": pos_row["event_time"],
            "is_positive": True,
        })

        # Add N random negative items (not in training, not positive)
        candidate_negatives = list(all_items - train_user_items - positive_items)
        n_actual = min(n_negatives, len(candidate_negatives))
        if n_actual > 0:
            neg_items = rng.choice(candidate_negatives, size=n_actual, replace=False)
            for neg_item in neg_items:
                rows.append({
                    "user_id": user_id,
                    "item_id": neg_item,
                    "hours": group["hours"].median(),  # placeholder
                    "event_time": group["event_time"].max(),
                    "is_positive": False,
                })

    return pd.DataFrame(rows)


print("Generating val candidates...")
candidates_val = simulate_candidates(val, train, catalog)
print(f"Val candidates: {len(candidates_val):,} rows ({candidates_val['user_id'].nunique():,} users)")
print(f"  Positive: {candidates_val['is_positive'].sum():,} | Negative: {(~candidates_val['is_positive']).sum():,}")

print("\nGenerating test candidates...")
candidates_test = simulate_candidates(test, train, catalog)
print(f"Test candidates: {len(candidates_test):,} rows ({candidates_test['user_id'].nunique():,} users)")
print(f"  Positive: {candidates_test['is_positive'].sum():,} | Negative: {(~candidates_test['is_positive']).sum():,}")

## 3. Feature Engineering

We call `build_features()` which orchestrates all feature computation.
**Critical:** all features are computed from `train` only — zero leakage from val/test.

The function:
1. Computes user aggregates from train
2. Computes item aggregates from train + catalog
3. Merges onto candidates + computes cross-features
4. Adds temporal features (days_since_last, hour cyclics)

In [ ]:
t0 = time.perf_counter()

df_val, feature_cols = build_features(candidates_val, train, catalog)
df_test, _ = build_features(candidates_test, train, catalog)

elapsed = time.perf_counter() - t0
print(f"Feature engineering: {elapsed:.1f}s")
print(f"\n{len(feature_cols)} features (after one-hot encoding category):")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")

print(f"\nVal feature matrix:  {df_val.shape}")
print(f"Test feature matrix: {df_test.shape}")

### Feature Correlation

We check for redundancy — highly correlated features don't add value and can be removed.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr = df_val[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

# Flag high correlations (>0.9)
high_corr = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        if abs(corr.iloc[i, j]) > 0.9:
            high_corr.append((feature_cols[i], feature_cols[j], corr.iloc[i, j]))

if high_corr:
    print(f"⚠ High correlations (>0.9):")
    for a, b, v in high_corr:
        print(f"  {a} ↔ {b}: {v:.3f}")
else:
    print("✓ No features with correlation > 0.9 — all bring unique signal.")

## 4. Train LightGBM LambdaRank

In [ ]:
X_train, y_train, q_train = prepare_ranking_data(df_val, feature_cols)
X_test, y_test, q_test = prepare_ranking_data(df_test, feature_cols)

print(f"Training set: {X_train.shape[0]:,} rows, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Query groups: {len(q_train):,} (train) | {len(q_test):,} (test)")
print(f"Avg group size: {q_train.mean():.0f} (train) | {q_test.mean():.0f} (test)")

In [ ]:
t0 = time.perf_counter()

model = train_ranker(
    X_train, y_train, q_train,
    X_test, y_test, q_test,
    feature_names=feature_cols,
    num_boost_round=500,
    early_stopping_rounds=30,
)

train_time = time.perf_counter() - t0
print(f"\nTraining completed in {train_time:.1f}s")
print(f"Best iteration: {model.best_iteration}")
print(f"Best NDCG@10 on val: {model.best_score.get('valid_0', {}).get('ndcg@10', 'N/A')}")

## 5. Evaluate

In [ ]:
catalog_size = len(catalog)
metrics = evaluate_ranker(model, X_test, y_test, q_test, catalog_size, k=10)

print("=" * 50)
print("LightGBM LambdaRank — Test Results")
print("=" * 50)
print(f"NDCG@10:           {metrics['ndcg_10']:.4f} ± {metrics['ndcg_10_std']:.4f}")
print(f"Catalog Coverage:  {metrics['catalog_coverage']:.2%}")
print(f"Users evaluated:   {metrics['num_users_evaluated']:,}")
print(f"Training time:     {train_time:.1f}s")
print(f"Catalog size:      {catalog_size:,}")

### Interpretation

- NDCG@10 measures ranking quality — how well the top-10 matches the user's true preferences.
- Catalog Coverage tells us whether the reranker recommends diverse items or only the head.
- The simulation uses random negatives — real retrieval candidates will produce different (better)
  NDCG because the negatives will be "harder" (items the retriever thought were relevant).

## 6. Feature Importance

In [ ]:
importance = model.feature_importance(importance_type="gain")
importance_df = (
    pd.DataFrame({"feature": feature_cols, "gain": importance})
    .sort_values("gain", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_df["feature"][::-1], importance_df["gain"][::-1])
ax.set_xlabel("Gain (feature importance)")
ax.set_title("Top 15 Features — LightGBM LambdaRank")
plt.tight_layout()
plt.show()

print("\nTop 15 features by gain:")
for _, row in importance_df.iterrows():
    print(f"  {row['feature']:<40s} {row['gain']:>12.0f}")

### Interpretation

Which features drive the ranking? Typically:
- **Item popularity** dominates (popular items are safer bets)
- **Cross-features** (category affinity, hours ratio) personalize the ranking
- **Temporal features** capture recency bias — recent activity matters more
- If a feature has near-zero gain, consider dropping it to reduce model size

## 7. Latency Breakdown

Measure how long each component takes — this feeds into ANALYSIS.md.

In [ ]:
# Simulate a single API request: user_id → features → predict
sample_user = df_test["user_id"].iloc[0]
sample_candidates = df_test[df_test["user_id"] == sample_user]

# Feature building (already done, but measure the predict step)
X_sample = sample_candidates[feature_cols].fillna(0).values.astype(np.float32)

n_runs = 100
times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    scores = model.predict(X_sample)
    top10 = np.argsort(scores)[::-1][:10]
    times.append(time.perf_counter() - t0)

avg_ms = np.mean(times) * 1000
p99_ms = np.percentile(times, 99) * 1000
print(f"LightGBM predict + top-10 sort:")
print(f"  Avg: {avg_ms:.2f} ms")
print(f"  P99: {p99_ms:.2f} ms")
print(f"  Candidates: {len(sample_candidates)}")

## 8. Export Model for API

Save LightGBM model + feature names so the FastAPI can load and use them.

In [ ]:
save_model(model, feature_cols)

print(f"Model saved:  models/ranker.txt")
print(f"Features saved: artifacts/feature_names.json")
print(f"\nFiles:")
for p in sorted(Path("models").glob("*")):
    print(f"  {p} ({p.stat().st_size / 1024:.1f} KB)")
for p in sorted(Path("artifacts").glob("*")):
    print(f"  {p} ({p.stat().st_size / 1024:.1f} KB)")

## 9. Summary

What we built:
- 14 features engineered from Steam interaction + catalog data
- LightGBM LambdaRank model trained with early stopping
- NDCG@10 + Catalog Coverage evaluation
- Feature importance analysis
- Latency benchmark
- Model export for API

**Next steps:**
1. Replace simulated candidates with real FAISS top-100 from Task 3 retriever
2. Tune hyperparameters (num_leaves, learning_rate, feature_fraction)
3. Add MMR re-ranking for diversity (bonus)
4. Integrate into FastAPI: `POST /recommendations/personalized/{user_id}`